# Train Nexo on Kaggle and store it on Hugging Face
Enable a GPU in **Settings → Accelerator → GPU**, attach a dataset containing a text corpus and `multimodal.jsonl`, and add a Kaggle Secret named `HF_TOKEN` with a Hugging Face write token.

In [ ]:
HF_REPO_ID = 'Radin132/nexo'
EPOCHS = 1
BATCH_SIZE = 2
CONTEXT_LENGTH = 256
IMAGE_SIZE = 128
PATCH_SIZE = 16
OUTPUT_DIR = '/kaggle/working/nexo-vision'

In [ ]:
import torch

assert torch.cuda.is_available(), 'Enable a GPU in Kaggle Settings first.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!git clone --depth 1 https://github.com/Radin-dev1/Nexo.git /kaggle/working/Nexo
%cd /kaggle/working/Nexo
!python -m pip install -q -e .

In [ ]:
from pathlib import Path

input_root = Path('/kaggle/input')
manifests = list(input_root.rglob('multimodal.jsonl'))
text_files = [p for p in input_root.rglob('*.txt') if p.stat().st_size > 0]
assert manifests, 'Attach a Kaggle dataset containing multimodal.jsonl.'
assert text_files, 'Attach a Kaggle dataset containing a non-empty .txt corpus.'
MANIFEST = str(manifests[0])
TEXT_DATA = str(text_files[0])
print('Manifest:', MANIFEST)
print('Tokenizer text:', TEXT_DATA)

In [ ]:
!python train_tokenizer.py --data {TEXT_DATA} --output /kaggle/working/nexo-tokenizer --vocab-size 32000

In [ ]:
!python train_multimodal.py \
  --manifest {MANIFEST} \
  --tokenizer /kaggle/working/nexo-tokenizer \
  --output {OUTPUT_DIR} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --context-length {CONTEXT_LENGTH} \
  --image-size {IMAGE_SIZE} \
  --patch-size {PATCH_SIZE}

In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
api = HfApi(token=hf_token)
api.create_repo(repo_id=HF_REPO_ID, repo_type='model', exist_ok=True)
api.upload_folder(
    repo_id=HF_REPO_ID,
    repo_type='model',
    folder_path=OUTPUT_DIR,
    commit_message='Upload Kaggle-trained Nexo multimodal model',
)
print(f'Published: https://huggingface.co/{HF_REPO_ID}')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(HF_REPO_ID, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    HF_REPO_ID, token=hf_token, trust_remote_code=True
)
print('Reloaded from Hugging Face:', model.config.model_type)